# **SDG Text Classification**
## BERT-base on SDG Knowledge Hub

### Objective
Clean the text, tokenize it, and fine-tune a model to predict one or more of the 17 SDGs.

### Pipeline
1. Load and clean the text.
2. Prepare labels and separate training, validation and test data.
3. Tokenize the text using the pretrained model's tokenizer.
4. Fine-tune, evaluate and save the classifier.


Dataset: [SDG Knowledge Hub](https://zenodo.org/records/7523032), Wulff and Meier, CC BY 4.0.

Knowledge Hub contains news and policy articles. These results describe article classification, not accuracy on scientific abstracts.

### 1. Environment Setup

#### Step 1. Install the required packages


In [ ]:
%pip install "transformers==4.48.3" "datasets==3.2.0" "accelerate==1.3.0" "torch>=2.6,<3" "pyarrow>=14,<21" pandas sentencepiece scikit-learn

#### Step 2. Import libraries and set the model

Using the English, uncased BERT-base checkpoint.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed
)

set_seed(42)
MODEL_NAME = "google-bert/bert-base-uncased"

### 2. Text Cleaning and Labels

#### Step 1. Load the dataset

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/thesis/SDG Knowledge Hub Dataset of SDG-labeled News Articles/sdg_knowledge_hub.csv")
sdg_columns = [f"SDG-{i:02d}" for i in range(1, 18)]

print("Articles:", len(df))

#### Step 2. Clean the text

We can keep punctuation and complete words because the pretrained tokenizer expects natural text. therefore we do not stem words or remove stopwords.

In [ ]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
df["text"] = df["text"].fillna("").apply(clean_text)
df = df[df["text"].ne("")].drop_duplicates(subset="text")

print(df["text"].iloc[0][:300])

#### Step 3. Prepare the multi-label targets

The dataset already has 17 SDG indicator columns. Keep them in SDG 1–17 order and convert them to floating-point values for binary cross-entropy.

In [ ]:
labels = df[sdg_columns].astype(float).values.tolist()
data = Dataset.from_dict({"text": df["text"].tolist(), "labels": labels})

print(data[0]["labels"])

#### Step 4. Separate training, validation and test data

Split the deduplicated articles into approximately 80% training, 10% validation and 10% test data. This simple random split is not multi-label stratified. Only validation data is used during training.

In [ ]:
split = data.train_test_split(test_size=0.2, seed=42)
held_out = split["test"].train_test_split(test_size=0.5, seed=42)
train_data = split["train"]
validation_data = held_out["train"]
test_data = held_out["test"]

print(len(train_data), len(validation_data), len(test_data))

### 3. Tokenization

#### Step 1. Load the tokenizer

Use the tokenizer paired with the pretrained model. Keep at most 512 tokens; longer texts are truncated in this baseline.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_text(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

#### Step 2. Tokenize each split

Tokenize in batches and keep the labels.

In [ ]:
train_data = train_data.map(tokenize_text, batched=True, remove_columns=["text"])
validation_data = validation_data.map(tokenize_text, batched=True, remove_columns=["text"])
test_data = test_data.map(tokenize_text, batched=True, remove_columns=["text"])

### 4. Model Fine-Tuning

#### Step 1. Load the classifier

Add 17 classification outputs. multi_label_classification uses binary cross-entropy so that each SDG can be predicted independently. The new classification head starts untrained.

In [ ]:
id2label = {i: f"SDG {i + 1}" for i in range(17)}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=17,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id={label: i for i, label in id2label.items()}
)

#### Step 2. Define classification metrics

Predict an SDG when its probability is at least 0.5. Calculate precision, recall and F1 for each text, then average over texts (average="samples"). Accuracy measures exact matches of the complete label set. Undefined precision or recall is set to zero. The threshold is a fixed baseline setting.

In [ ]:
def compute_metrics(prediction):
    probabilities = torch.sigmoid(torch.tensor(prediction.predictions)).numpy()
    predicted_labels = (probabilities >= 0.5).astype(int)
    true_labels = prediction.label_ids

    return {
        "precision": precision_score(true_labels, predicted_labels, average="samples", zero_division=0),
        "recall": recall_score(true_labels, predicted_labels, average="samples", zero_division=0),
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "f1": f1_score(true_labels, predicted_labels, average="samples", zero_division=0)
    }

#### Step 3. Set training parameters

Start with a learning rate of 2e-5, batch size of 4 and three epochs. Evaluate after each epoch and keep the checkpoint with the lowest validation loss. These values are starting settings, not tuned results.

In [ ]:
training_args = TrainingArguments(
    output_dir="bert_knowledge_hub/checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=1,
    report_to="none",
    dataloader_pin_memory=False,
    seed=42
)

#### Step 4. Fine-tune the model

The Trainer handles batching, optimization and validation. `DataCollatorWithPadding` pads texts to the longest sequence in each batch.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=validation_data,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

### 5. Evaluation and Saving

#### Step 1. Evaluate on the test set

Evaluate once after fine-tuning.

In [ ]:
test_results = trainer.evaluate(test_data, metric_key_prefix="test")

In [ ]:
test_results

#### Step 2. Save the classifier

Save the model and tokenizer together so that the same text processing can be used when making predictions.

In [ ]:
trainer.save_model("/content/drive/MyDrive/thesis/bert_knowledge_hub/model")
tokenizer.save_pretrained("/content/drive/MyDrive/thesis/bert_knowledge_hub/model")